# Basal forebrain functional connectivity gradients in Parkinson's disease
## Ignatavicius et al. 
Adapted from Chakraborty, S., Haast, R.A.M., Onuska, K.M. et al. Multimodal gradients of basal forebrain connectivity across the neocortex. Nat Commun 15, 8990 (2024). https://doi.org/10.1038/s41467-024-53148-x 

### Sections
1. Imports and configuration
2. Load functional matrices and build whole-sample group average
3. Gradient decomposition 
4. Gradient-weighted cortical expression 
5. BF subregion analysis: Ch4 vs Ch1-3 (distribution + surrogate tests)
6. Subject-level gradient alignment and cortical expression
7. PD vs control comparisons
8. Check robustness of HC cortical expression template


## 1. Import packages and set paths

In [ ]:
import os, re
from glob import glob
import numpy as np
import pandas as pd
import scipy.io as sio
import h5py
from scipy.stats import pearsonr, zscore, ranksums, ttest_ind
from scipy.spatial.distance import cdist, cosine, euclidean
from sklearn.preprocessing import MinMaxScaler
import nibabel as nib
from nilearn.plotting import plot_glass_brain
from brainspace.gradient import GradientMaps
from brainspace.gradient.kernels import compute_affinity
from brainspace.datasets import load_fsa5
from brainspace.utils.parcellation import map_to_labels
from brainspace.plotting import plot_hemispheres
from brainspace.null_models.variogram import SurrogateMaps
from neuromaps import images
from neuromaps.datasets import fetch_fsaverage
from surfplot import Plot
import pingouin as pg
from pingouin import ttest
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import matplotlib, matplotlib.pyplot as plt, matplotlib.cm
from matplotlib.colors import ListedColormap
import seaborn as sns
matplotlib.rcParams['font.family'] = 'Arial'
SEED = 42
np.random.seed(SEED)
print('Imports done.')


### Set paths 

In [ ]:
main_path    = '/path/to/directory'              
results_path = '/path/to/output_dir'   
fig_path     = '/path/to/output_dir/figures/'
COV_FILE     = '/path/to/subject_data' # subject demographics 
CTRL_DIR     = '/path/to/control_dir' # where FC matrix lives
PD_DIR       = '/path/to/pd_dir'      # where FC matrix lives
BF_SEED      = os.path.join(main_path, 'BF_seed_2mm.nii.gz')
BF_ATLAS     = os.path.join(main_path, 'BF_Masks', 'BF_masked_fullB_2mm.dseg.nii.gz')
BF_SURF_LH   = os.path.join(main_path, 'seed-BASF.L.bin.fsa5.shape.gii')
BF_SURF_RH   = os.path.join(main_path, 'seed-BASF.R.bin.fsa5.shape.gii')
os.makedirs(results_path, exist_ok=True)
os.makedirs(fig_path, exist_ok=True)
n_components = 16
N_SURR = 1000
print(f'Results -> {results_path}')


## 2. Load FC matrices and build group-averaged matrix
Loads all HC + PD per-subject BF×400-parcel FC matrices. The **whole-cohort** mean is used as input to the gradient decomposition.

In [ ]:
ctrl_files = sorted(glob(os.path.join(CTRL_DIR, "*BFvox_x_SchaeferSurf_FC.mat")))
pd_files   = sorted(glob(os.path.join(PD_DIR,   "*BFvox_x_SchaeferSurf_FC.mat")))

print(f'Found Control files: {len(ctrl_files)}')
print(f'Found PD files:      {len(pd_files)}')

mat_files = ctrl_files + pd_files
group     = np.array(['Control']*len(ctrl_files) + ['PD']*len(pd_files), dtype=object)

def extract_sub_id(path):
    m = re.search(r'(sub-\d+)', os.path.basename(path))
    return m.group(1) if m else os.path.basename(path).split('_')[0]

def load_fc(path):
    try:
        with h5py.File(path, 'r') as mat_file:
            return np.array(mat_file['R']).T
    except OSError:
        mat = sio.loadmat(path)
        return mat['R']

sub_id = np.array([extract_sub_id(f) for f in mat_files], dtype=object)

fc_matrices, keep = [], []
expected_shape = None

for f in mat_files:
    try:
        fc = load_fc(f)
    except Exception as e:
        print(f"Skipping {os.path.basename(f)} (load error: {e})")
        keep.append(False)
        continue

    if expected_shape is None:
        expected_shape = fc.shape
        print('Expected FC shape:', expected_shape)

    if fc.shape != expected_shape:
        print(f'Skipping {os.path.basename(f)} (shape mismatch: {fc.shape})')
        keep.append(False)
        continue

    fc_matrices.append(fc)
    keep.append(True)

keep     = np.array(keep, dtype=bool)
fc_array = np.stack(fc_matrices, axis=0)

group    = group[keep]
sub_id   = sub_id[keep]

isPD     = group == 'PD'
isCON    = group == 'Control'

print(f'Loaded {fc_array.shape[0]} subjects. FC array: {fc_array.shape}')
print(f'Groups: {dict((g, int(np.sum(group==g))) for g in np.unique(group))}')

group_avg_fc = np.mean(fc_array, axis=0)
print('Whole-cohort group-average FC shape:', group_avg_fc.shape)
print(f"Mean FC: {np.nanmean(group_avg_fc):.4f}")
print(f"Min FC:  {np.nanmin(group_avg_fc):.4f}")
print(f"Max FC:  {np.nanmax(group_avg_fc):.4f}")

pd.DataFrame(np.abs(group_avg_fc)).to_csv(
    results_path + 'Func_abs-corr-avgFC.csv',
    index=False
)

In [ ]:
f, ax = plt.subplots(1,1,dpi=100)
im = ax.imshow(group_avg_fc)
ax.set_title('Connectivity Matrix {}'.format(group_avg_fc.shape), fontsize=18)
cbar = plt.colorbar(im)
cbar.set_label("Pearson's correlation", fontsize=18)
cbar.ax.tick_params(labelsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.savefig(fig_path + 'Parcellated_func-ConnMatrix.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
f, ax = plt.subplots(1,1,dpi=100)
ax.imshow(group_avg_fc)
ax.set_xticks([]); ax.set_yticks([])
plt.savefig(fig_path + 'Func_ConnMatrix_forSuppFig.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
corr_avgFC = group_avg_fc.copy()
corr_avgFC += 1
corr_avgFC[np.isnan(corr_avgFC)] = 1
pd.DataFrame(corr_avgFC).to_csv(results_path + 'Func_corr-avgFC.csv', index=False)


## 3. Gradient Decomposition
Diffusion map embedding on the normalised angle affinity matrix of the whole-sample FC.

In [ ]:
sm_matrix = compute_affinity(corr_avgFC, kernel='normalized_angle')
f, ax = plt.subplots(1,1,dpi=100)
im = ax.imshow(sm_matrix)
ax.set_title('Affinity Matrix {}'.format(sm_matrix.shape), fontsize=10)
cbar = plt.colorbar(im)
cbar.set_label('Similarity', fontsize=10)
cbar.ax.tick_params(labelsize=10)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.savefig(fig_path + 'Parcellated_func-SimMatrix.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
f, ax = plt.subplots(1,1,dpi=100)
ax.imshow(sm_matrix)
ax.set_xticks([]); ax.set_yticks([])
plt.savefig(fig_path + 'Func_SimMatrix_suppFig.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()


In [ ]:
gm = GradientMaps(n_components=n_components, kernel='normalized_angle', approach='dm', random_state=SEED)
print(gm.fit(corr_avgFC))

pd.DataFrame(gm.gradients_).to_csv(results_path + f'Func_gradients-{n_components}_seed-BF_2mm.csv', index=False)
seed_nib = nib.load(BF_SEED)
seed_vol = seed_nib.get_fdata()
indices = np.array(np.where(seed_vol > 0)).T
for gradi in range(4):
    grad_vol = np.zeros(seed_vol.shape)
    grad_vol[indices[:,0],indices[:,1],indices[:,2]] = gm.gradients_[:,gradi]
    nib.save(nib.Nifti1Image(grad_vol, seed_nib.affine, seed_nib.header),
             results_path + f'Func_gradient_{gradi+1}.nii.gz')
print('Saved G1-G4 NIfTI.')


### G1 voxel bar plot

In [ ]:
G1 = gm.gradients_[:,0]
f, ax = plt.subplots(1,1,dpi=100)
ax.imshow(G1.reshape([len(G1),1]).T, cmap='seismic', aspect=30)
plt.tight_layout()
ax.set_xticks([]); ax.set_yticks([])
plt.savefig(fig_path + 'Parcellated_func-G1.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()



### Project BF gradients into 3D voxel space

In [ ]:
fname = results_path + 'Func_gradient_{0}.nii.gz'
G_idx = {}
G_values = {}
for g in range(1,5):
    G_nii = nib.load(fname.format(g)).get_fdata()
    G_idx[g] = np.argwhere(G_nii)
    G_values[g] = G_nii[G_idx[g][:,0],G_idx[g][:,1],G_idx[g][:,2]].flatten()
colors_3d = ['seismic','seismic','PiYG','bwr','PRGn','coolwarm']
for g in range(1,5):
    fig = plt.figure(figsize=(10,10))
    ax = fig.add_subplot(projection='3d')
    ax.dist=15
    ax.set_axis_off()
    ax.scatter(G_idx[g][:,0],G_idx[g][:,1],G_idx[g][:,2],
               s=120,alpha=1,cmap=colors_3d[g],
               vmin=G_values[g].min(),vmax=G_values[g].max(),c=G_values[g])
    sm_c = plt.cm.ScalarMappable(cmap=colors_3d[g],norm=plt.Normalize(vmin=G_values[g].min(),vmax=G_values[g].max()))
    sm_c.set_array([])
    plt.colorbar(sm_c,ax=ax,shrink=0.4)
    plt.savefig(fig_path + f'Func_Gradient_{g}_scatterplot.png', dpi=300, bbox_inches='tight', transparent=True)
    plt.show()


### Glass brain visualisation

In [ ]:
colors_gb = ['seismic','PiYG','bwr','PRGn']
for gradi in range(4):
    grad_nib = nib.load(results_path + f'Func_gradient_{gradi+1}.nii.gz')
    color = matplotlib.cm.get_cmap(colors_gb[gradi])
    plot_glass_brain(grad_nib, colorbar=True, display_mode='lyrz', plot_abs=False, cmap=color)
    plt.savefig(fig_path + f'Func_Gradient_{gradi+1}.png', dpi=300)
    plt.show()


### Variance explained

In [ ]:
fig, ax = plt.subplots(1, figsize=(5,4))
ax.scatter(range(gm.lambdas_.size), gm.lambdas_)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlabel('Component Number', fontsize=10)
ax.set_ylabel('Eigenvalues', fontsize=10)
plt.savefig(fig_path + f'Func_Gradient_eigenvalue_gm-{n_components}.png', dpi=300, bbox_inches='tight')
plt.show()
variance = gm.lambdas_ / np.sum(gm.lambdas_)
fig, ax = plt.subplots(1, figsize=(5,4))
x = np.arange(n_components)
color = np.where(x < 1, 'red', 'C0')
ax.scatter(x, variance, c=color)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlabel('Component Number', fontsize=10)
ax.set_ylabel('Variance explained', fontsize=10)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.savefig(fig_path + f'Func_Gradient_variance_gm-{n_components}.png', dpi=300, bbox_inches='tight')
plt.show()
pd.DataFrame(variance).to_csv(results_path + f'Func_gradients_variance_gm-{n_components}.csv', index=False)
cum_var = np.cumsum(gm.lambdas_) / np.sum(gm.lambdas_)
fig, ax = plt.subplots(1, figsize=(5,4))
ax.scatter(range(cum_var.size), cum_var)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlabel('Component Number', fontsize=10)
ax.set_ylabel('Cumulative variance', fontsize=10)
plt.savefig(fig_path + f'Func_Gradient_cumulative-variance_gm-{n_components}.png', dpi=300, bbox_inches='tight')
plt.show()


## 4. Gradient-Weighted Cortical Projection (Group Average)
Weight each FC row by the BF voxel gradient score, average across BF voxels -> 400-parcel cortical map.

In [ ]:
G_Ctx = {}
for g in range(4):
    G_Ctx[g] = np.zeros(corr_avgFC.shape)
    for i in range(len(gm.gradients_[:,g])):
        G_Ctx[g][i,:] = corr_avgFC[i,:] * gm.gradients_[i,g]
np.savez(results_path + 'Func_Gradient-weighted_CorticalConnectivity.npz',
         G1_Ctx=G_Ctx[0], G2_Ctx=G_Ctx[1], G3_Ctx=G_Ctx[2], G4_Ctx=G_Ctx[3])
G1_Ctx = G_Ctx[0]
f, ax = plt.subplots(1,1,dpi=100)
ax.imshow(G1_Ctx, cmap='seismic')
ax.set_xticks([]); ax.set_yticks([])
plt.savefig(fig_path + 'Parcellated_func-G1-weightedConnMatrix.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
Gfc = {}
for g in range(4):
    Gfc[g] = np.nanmean(G_Ctx[g], axis=0).reshape([400,1])
Gfc1 = Gfc[0]
f, ax = plt.subplots(1,1,dpi=100)
ax.imshow(Gfc1.T, cmap='seismic', aspect=30)
ax.set_xticks([]); ax.set_yticks([])
plt.savefig(fig_path + 'Parcellated_func-cortical-G1.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()


In [ ]:
surf_lh, surf_rh = load_fsa5()
surf_labels_lh = nib.freesurfer.read_annot('/path/to/fsa5/lh.Schaefer2018_400Parcels_7Networks_order.annot')[0]
surf_labels_rh = nib.freesurfer.read_annot('/path/to/fsa5/rh.Schaefer2018_400Parcels_7Networks_order.annot')[0]
surf_labels_rh[surf_labels_rh != 0] += 200
surf_labels = np.concatenate([surf_labels_lh, surf_labels_rh])
mask = surf_labels != 0
G_cortex = {}
for g in range(4):
    G_cortex[g] = map_to_labels(Gfc[g].reshape([400]), surf_labels, mask=mask, fill=np.nan)
colors_surf = ['seismic','PiYG','bwr','PRGn']
for gradi in range(4):
    plot_hemispheres(surf_lh, surf_rh, array_name=[G_cortex[gradi]], size=(400,400),
                    layout_style='grid', label_text=[f'Gradient-{gradi+1}'],
                    cmap=[colors_surf[gradi],colors_surf[gradi]], color_bar=False,
                    color_range='sym', zoom=1.2, embed_nb=True, screenshot=True,
                    filename=fig_path+f'Func_Grad-{gradi+1}_weighted_cortex.png')
for gradi in range(4):
    fig_s = plot_hemispheres(surf_lh, surf_rh, array_name=[G_cortex[gradi]],
                             label_text=[f'Gradient-{gradi+1}'], size=(800,200),
                             cmap=[colors_surf[gradi]], color_range='sym', color_bar=True, embed_nb=True)
    display(fig_s)
for g in range(4):
    for hemi, sl, offset in [('L', surf_labels_lh, 0), ('R', surf_labels_rh, 10242)]:
        gii = nib.gifti.GiftiImage()
        gii.add_gifti_data_array(nib.gifti.GiftiDataArray(
            G_cortex[g][offset:offset+10242].astype(np.float32)))
        nib.save(gii, results_path+f'Func_Gradient-{g+1}_weighted_{hemi}_fsa-10k.gii')


In [ ]:
BF_lh = images.load_gifti(BF_SURF_LH)
BF_rh = images.load_gifti(BF_SURF_RH)
BF_lh_data = BF_lh.agg_data()
BF_rh_data = BF_rh.agg_data()
BF_data = np.concatenate((BF_lh_data, BF_rh_data))
scaler = MinMaxScaler()
BF_scaled_data = scaler.fit_transform(BF_data.reshape(-1,1))
BF_scaled_data = np.where(BF_scaled_data == 0.5, 1, BF_scaled_data)
surfaces = fetch_fsaverage(density='10k')
lh, rh = surfaces['inflated']
for g_idx, cmap_name in enumerate(['seismic','PiYG']):
    p = Plot(lh, rh, size=(800,600), zoom=1.5, brightness=.8)
    p.add_layer(G_cortex[g_idx], cmap=cmap_name, cbar=True)
    p.add_layer(BF_scaled_data.reshape([20484]), cmap='binary_r', cbar=False)
    fig_p = p.build()
    fig_p.savefig(fig_path+f'Func_G{g_idx+1}-weighted_cortex.png', dpi=300, transparent=True)
    plt.show()


## 5. BF subregion analysis: Ch4 vs Ch123
Distribution of gradient values across BF subregions (Zaborszky atlas) + surrogate tests

In [ ]:
subBF_nib = nib.load(BF_ATLAS)
subBF_vol = subBF_nib.get_fdata()
grad_1_nib = nib.load(results_path + 'Func_gradient_1.nii.gz')
grad_1_vol = grad_1_nib.get_fdata()
grad_2_nib = nib.load(results_path + 'Func_gradient_2.nii.gz')
grad_2_vol = grad_2_nib.get_fdata()
df_sub = pd.DataFrame(dict(
    labels=subBF_vol[subBF_vol!=0].flatten(),
    funcG1=grad_1_vol[subBF_vol!=0].flatten(),
    funcG2=grad_2_vol[subBF_vol!=0].flatten()))
pd.DataFrame(df_sub).to_csv(results_path + 'Func_grad-atlas2labels.csv', index=False)
bf_idx = np.argwhere(subBF_vol)
bf_values = subBF_vol[bf_idx[:,0],bf_idx[:,1],bf_idx[:,2]].flatten()
atlas_cols = ListedColormap(["#7D3C98", "#2196A6"])
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(projection='3d')
ax.dist=10
ax.set_axis_off()
ax.scatter(bf_idx[:,0],bf_idx[:,1],bf_idx[:,2],s=120,alpha=1,cmap=atlas_cols,c=bf_values)
plt.show()


In [ ]:
grad_1_nib = nib.load(results_path +'Func_gradient_1.nii.gz')
grad_1_vol = grad_1_nib.get_fdata()

df = pd.DataFrame(
    dict(
        labels=subBF_vol[subBF_vol!=0].flatten(),
        funcG1=grad_1_vol[subBF_vol!=0].flatten(),
        funcG2=grad_2_vol[subBF_vol!=0].flatten()
    )
)

labels=subBF_vol[subBF_vol!=0].flatten()

In [ ]:
pd.DataFrame(df).to_csv(results_path + f'Func_grad-atlas2labels.csv', index=False)
labels = ['Ch123', 'Ch4a/Ch4p']
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable

label_map = {1: "Ch123", 2: "Ch4a/Ch4p"}
df["labels"] = df["labels"].map(label_map)

labels_order = ["Ch123", "Ch4a/Ch4p"]
palette = ["#7D3C98", "#2196A6"]

fig, ax = plt.subplots(figsize=(8, 5))

sns.stripplot(
    data=df,
    x="labels",
    y="funcG1",
    order=labels_order,
    palette=palette,
    size=6,
    alpha=0.6,
    ax=ax
)

# cosmetics
ax.set_xlabel("BF subregions based on stereotactic atlas", fontsize=18)
ax.set_ylabel("")
ax.set_xticklabels(labels_order, fontsize=14, weight=600)
ax.set_yticks([])
sns.despine()

cmap = plt.get_cmap("seismic")
norm = plt.Normalize(df["funcG1"].min(), df["funcG1"].max())
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    location="left",
    ticks=[df["funcG1"].min(), df["funcG1"].max()]
)

cbar.ax.set_yticklabels(["−", "+"], fontsize=14, weight=600)
cbar.ax.set_ylabel("Gradient Values", fontsize=14, weight=600)
cbar.ax.set_title("G1", fontsize=12, weight=900)

print(df["labels"].value_counts())


### Simple t-test: Ch1-3 vs Ch4

In [ ]:
df_atlas = df  
ch123 = df_atlas.loc[df_atlas['labels']=='Ch123','funcG1'].values
ch4ap = df_atlas.loc[df_atlas['labels']=='Ch4a/Ch4p','funcG1'].values
ttest_emp = ttest(ch123, ch4ap)
print(ttest_emp)
print(f"Ch123:    mean={ch123.mean():.4f}, SD={ch123.std():.4f}, n={len(ch123)}")
print(f"Ch4a/Ch4p: mean={ch4ap.mean():.4f}, SD={ch4ap.std():.4f}, n={len(ch4ap)}")

### Surrogate maps (variogram-based spatial permutation)
Tests whether empirical variance/SD/mean differences between Ch4 and Ch1-3 exceed the null distribution,
preserving spatial autocorrelation.

In [ ]:
# Load seed mask
mask = nib.load('/path/to/BF_seed_2mm.nii.gz')
mask = mask.get_fdata()
ind  = np.argwhere(mask)
print("mask nonzero voxels:", mask.sum())
print("ind voxels:", ind.shape[0])

In [ ]:
# Load BASF atlas
atlas     = nib.load('/path/to/BF_masked_fullB_2mm.dseg.nii.gz')
atlas     = atlas.get_fdata()
atlas     = atlas[atlas!=0]

# load data and atlas
df = pd.read_csv(results_path + f'Func_grad-atlas2labels.csv')

funcG1 = df['funcG1'].values

from sklearn.preprocessing import MinMaxScaler
# min-max scale, for compatability with non-resampled surrogate maps
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(funcG1[:,np.newaxis])

In [ ]:
# Plot data
colors = ListedColormap(['red', 'green'])

fig = plt.figure(figsize=(15,5))
ax = fig.add_subplot(131, projection='3d')
ax.dist=6
ax.set_axis_off()
    
ax.scatter(
    ind[:,0],
    ind[:,1],
    ind[:,2],
    s=60, alpha=1,
    cmap=colors,
    c=atlas
)

ax1 = fig.add_subplot(132, projection='3d')
ax1.dist=6
ax1.set_axis_off()

ax1.scatter(
    ind[:,0],
    ind[:,1],
    ind[:,2],
    s=50, alpha=1,
    cmap='seismic',
    c=data_scaled[:,0]
)

ax2 = fig.add_subplot(133, box_aspect=2)
sns.scatterplot(x=atlas, y=data_scaled[:,0], hue=atlas, palette={1: 'red', 2: 'green'}, ax=ax2)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_ylabel('Gradient values', weight='bold')
ax2.set_xlabel('Atlas label', weight='bold')
ax2.set_xlim((0,3))

plt.show()

In [ ]:
# Calculate distance matrix for BASF voxels
dist = cdist(ind, ind)

# compute variograms
vg = SurrogateMaps(
    resample=False,
    pv=60,
    random_state=1234
).fit(dist)

# Calculate empirical variogram
emp_var = vg.compute_variogram(data_scaled[:,0])
emp_var, u0 = vg.smooth_variogram(emp_var, return_h=True)

In [ ]:
# Then use the fitted SurrogateMaps function to generate surrogate map
nsurr = 1000
surr = vg.randomize(data_scaled[:,0], n_rep=nsurr)

# Compute surrogate map variograms
surr_var = np.empty((nsurr, len(emp_var)))
for i in range(nsurr):
    tmp = vg.compute_variogram(np.array(surr[i,:]))
    surr_var[i] = vg.smooth_variogram(tmp)

# # Create plot for inspecting fit of surrogate models
# Plot empirical variogram
fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.12, 0.15, 0.8, 0.77])
ax.autoscale(axis='y', tight=True)

ax.scatter(u0, emp_var, s=20, facecolor='none', edgecolor='k',
           marker='o', lw=1, label='Empirical')

# Plot surrogate maps' variograms
mu = surr_var.mean(axis=0)
sigma = surr_var.std(axis=0)
ax.fill_between(u0, mu-sigma, mu+sigma, facecolor='#377eb8',
                edgecolor='none', alpha=0.3)
ax.plot(u0, mu, color='#377eb8', label='SA-preserving', lw=1)

# Make plot nice
leg = ax.legend(loc=0)
leg.get_frame().set_linewidth(0.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlabel("Spatial separation distance", weight='bold')
ax.set_ylabel("Variance", weight='bold')

plt.show()

In [ ]:
# Plot surrogate map example
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(projection='3d')
ax.dist=6
ax.set_axis_off()

ax.scatter(
    ind[:,0],
    ind[:,1],
    ind[:,2],
    s=50, alpha=1,
    cmap='seismic',
    c=surr[0,:]
)

In [ ]:
# Calculate variability for empirical data
x1 = data_scaled[atlas==1]
x2 = data_scaled[atlas==2]

# Standard deviation
std1 = np.std(x1)
std2 = np.std(x2)
emp_std_diff = std1-std2

# Variance
var1 = np.var(x1)
var2 = np.var(x2)
emp_var_diff = var1-var2

# Coefficient of variation
cov = lambda x: np.std(x, ddof=1) / np.mean(x) * 100
cov1 = cov(x1)
cov2 = cov(x2)
emp_cov_diff = cov1-cov2

print(emp_var_diff)
print(emp_cov_diff)

# Calculate surrogate variability
surr_std_diff = np.empty((nsurr, 1))
surr_var_diff = np.empty((nsurr, 1))
surr_cov_diff = np.empty((nsurr, 1))

for i in range(nsurr):
    x1 = surr[i,atlas==1]
    x2 = surr[i,atlas==2]
    
    # Standard deviation
    std1 = np.std(x1)
    std2 = np.std(x2)
    surr_std_diff[i] = std1-std2
    
    # Variance
    var1 = np.var(x1)
    var2 = np.var(x2)
    surr_var_diff[i] = var1-var2
    
    # Coefficient of variation
    cov = lambda x: np.std(x, ddof=1) / np.mean(x) * 100
    cov1 = cov(x1)
    cov2 = cov(x2)
    surr_cov_diff[i] = cov1-cov2

# Plot 
fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.12, 0.15, 0.8, 0.77])
ax.autoscale(axis='y', tight=True)

ax.hist(surr_var_diff, bins=25, density=True, alpha=0.5, color=(.8, .8, .8))
ax.axvline(emp_var_diff, lw=2, ls='--', color='k')
ax.set_title('Variance difference')

pvalue = 1-(np.sum(surr_var_diff>emp_var_diff)/nsurr)
print(f'Pvalue = {pvalue}')

In [ ]:
# Plot 
fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.12, 0.15, 0.8, 0.77])
ax.autoscale(axis='y', tight=True)

ax.hist(surr_cov_diff, bins=25, density=True, alpha=0.5, color=(.8, .8, .8))
ax.axvline(emp_cov_diff, lw=2, ls='--', color='k')
ax.set_title('Coefficient of variation difference')

pvalue = 1-(np.sum(surr_cov_diff>emp_cov_diff)/nsurr)
print(f'Pvalue = {pvalue}')

In [ ]:
# Plot 
fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.12, 0.15, 0.8, 0.77])
ax.autoscale(axis='y', tight=True)

ax.hist(surr_std_diff, bins=25, density=True, alpha=0.5, color=(.8, .8, .8))
ax.axvline(emp_std_diff, lw=2, ls='--', color='k')
ax.set_title('Standard deviation difference')

pvalue = 1-(np.sum(surr_std_diff>emp_std_diff)/nsurr)
print(f'Pvalue = {pvalue}')

In [ ]:
print(surr_var_diff.mean())
print(surr_cov_diff.mean())

In [ ]:
# Calculate variability for empirical data
x2 = data_scaled[atlas==1]
x1 = data_scaled[atlas==2]

# Standard deviation
std1 = np.std(x1)
std2 = np.std(x2)
emp_std_diff = std1-std2

# Variance
var1 = np.var(x1)
var2 = np.var(x2)
emp_var_diff = var1-var2

# Coefficient of variation
cov = lambda x: np.std(x, ddof=1) / np.mean(x) * 100
cov1 = cov(x1)
cov2 = cov(x2)
emp_cov_diff = cov1-cov2

print(emp_var_diff)
print(emp_cov_diff)

# Calculate surrogate variability
surr_std_diff = np.empty((nsurr, 1))
surr_var_diff = np.empty((nsurr, 1))
surr_cov_diff = np.empty((nsurr, 1))

for i in range(nsurr):
    x2 = surr[i,atlas==1]
    x1 = surr[i,atlas==2]
    
    # Standard deviation
    std1 = np.std(x1)
    std2 = np.std(x2)
    surr_std_diff[i] = std1-std2
    
    # Variance
    var1 = np.var(x1)
    var2 = np.var(x2)
    surr_var_diff[i] = var1-var2
    
    # Coefficient of variation
    cov = lambda x: np.std(x, ddof=1) / np.mean(x) * 100
    cov1 = cov(x1)
    cov2 = cov(x2)
    surr_cov_diff[i] = cov1-cov2

# Plot 
fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.12, 0.15, 0.8, 0.77])
ax.autoscale(axis='y', tight=True)

ax.hist(surr_var_diff, bins=25, density=True, alpha=0.5, color=(.8, .8, .8))
ax.axvline(emp_var_diff, lw=2, ls='--', color='k')
ax.set_title('Variance difference')

pvalue = 1-(np.sum(surr_var_diff>emp_var_diff)/nsurr)
print(f'Pvalue = {pvalue}')

In [ ]:
# Plot 
fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.12, 0.15, 0.8, 0.77])
ax.autoscale(axis='y', tight=True)

ax.hist(surr_cov_diff, bins=25, density=True, alpha=0.5, color=(.8, .8, .8))
ax.axvline(emp_cov_diff, lw=2, ls='--', color='k')
ax.set_title('Coefficient of variation difference')

pvalue = 1-(np.sum(surr_cov_diff>emp_cov_diff)/nsurr)
print(emp_cov_diff)
print(f'Pvalue = {pvalue}')


In [ ]:
# Plot 
fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.12, 0.15, 0.8, 0.77])
ax.autoscale(axis='y', tight=True)

ax.hist(surr_std_diff, bins=25, density=True, alpha=0.5, color=(.8, .8, .8))
ax.axvline(emp_std_diff, lw=2, ls='--', color='k')
ax.set_title('Standard deviation difference')

pvalue = 1-(np.sum(surr_std_diff>emp_std_diff)/nsurr)
print(f'Pvalue = {pvalue}')

In [ ]:
# Calculate mean diff for empirical data
x1 = data_scaled[atlas==1]
x2 = data_scaled[atlas==2]

# Mean 
mean1 = np.mean(x1)
mean2 = np.mean(x2)
emp_mean_diff = mean1-mean2

print(emp_mean_diff)

# Calculate surrogate mean diff
surr_mean_diff = np.empty((nsurr, 1))

for i in range(nsurr):
    x1 = surr[i,atlas==1]
    x2 = surr[i,atlas==2]
    
    # Standard deviation
    mean1 = np.mean(x1)
    mean2 = np.mean(x2)
    surr_mean_diff[i] = mean1-mean2

# Plot 
fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.12, 0.15, 0.8, 0.77])
ax.autoscale(axis='y', tight=True)

ax.hist(surr_mean_diff, bins=25, density=True, alpha=0.5, color=(.8, .8, .8))
ax.axvline(emp_mean_diff, lw=2, ls='--', color='k')
ax.set_title('Mean difference')

pvalue = 1-(np.sum(surr_mean_diff>emp_mean_diff)/nsurr)
print(f'Pvalue = {pvalue}')

## 6. Subject-level gradient alignment and cortical expression
1. Align each subject to whole-sample gradient template (Procrustes)
2. Compute subject-level gradient-weighted cortical expression (Gfc_sub)
3. Plot HC mean and PD mean cortical expression maps
4. Compute similarity: subject Gfc vs HC mean Gfc (normative cortical expression template)

In [ ]:
G_ref  = gm.gradients_.copy()  # whole-sample template
G_refK = G_ref[:,:4]
nSub, nBF, nParc = fc_array.shape
K = 4
G_ind = np.zeros((nSub, nBF, K), dtype=np.float32)
for s in range(nSub):
    fc_s = fc_array[s].copy().astype(float)
    fc_s += 1
    fc_s[np.isnan(fc_s)] = 1
    gm_s = GradientMaps(n_components=K, kernel='normalized_angle', approach='dm',
                        alignment='procrustes', random_state=SEED)
    gm_s.fit(fc_s, reference=G_refK)
    G_ind[s] = gm_s.aligned_[:,:K]
G_mean_PD  = np.mean(G_ind[isPD],  axis=0)
G_mean_CON = np.mean(G_ind[isCON], axis=0)
print(f'G_ind shape: {G_ind.shape}')


In [ ]:
# Plot HC and PD mean gradient in BF space (3D scatter)
K=1
def get_sym_limits(a, b, prc=(2,98)):
    x = np.concatenate([a.flatten(),b.flatten()])
    x = x[np.isfinite(x)]
    lim = max(abs(np.percentile(x,prc[0])), abs(np.percentile(x,prc[1])))
    return -lim, lim
def plot_bf_3d(vec, title, out_png, cmap_name='seismic', vmin=None, vmax=None):
    G_nii = nib.load(fname.format(1)).get_fdata()
    bf_coords = np.argwhere(G_nii != 0)
    if vmin is None: vmin=np.percentile(vec,2)
    if vmax is None: vmax=np.percentile(vec,98)
    vabs = max(abs(vmin),abs(vmax))
    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(projection='3d')
    ax.dist=15
    ax.set_axis_off()
    sc = ax.scatter(bf_coords[:,0],bf_coords[:,1],bf_coords[:,2],
                    s=80,alpha=1,cmap=cmap_name,vmin=-vabs,vmax=vabs,c=vec)
    plt.colorbar(sc,ax=ax,shrink=0.4)
    ax.set_title(title,fontsize=11)
    plt.savefig(out_png,dpi=300,bbox_inches='tight',transparent=True)
    plt.show()
for g in range(K):
    vmin_g, vmax_g = get_sym_limits(G_mean_PD[:,g], G_mean_CON[:,g])
    plot_bf_3d(G_mean_CON[:,g], f'G{g+1} HC mean', fig_path+f'CON_G{g+1}_BF_3d.png',
               vmin=vmin_g, vmax=vmax_g)
    plot_bf_3d(G_mean_PD[:,g],  f'G{g+1} PD mean', fig_path+f'PD_G{g+1}_BF_3d.png',
               vmin=vmin_g, vmax=vmax_g)


In [ ]:
# Subject-level Gfc (gradient-weighted cortical expression)
K=1
Gfc_sub = {}
for g in range(K):
    Gfc_sub[g] = np.zeros((nSub, nParc), dtype=np.float32)
    for s in range(nSub):
        fc_s = fc_array[s].copy().astype(float)
        fc_s += 1
        fc_s[np.isnan(fc_s)] = 1
        G_Ctx_s = fc_s * G_ind[s,:,g:g+1]
        Gfc_sub[g][s] = np.nanmean(G_Ctx_s, axis=0)
# HC normative cortical expression template
Gfc_mean_CON = {g: np.nanmean(Gfc_sub[g][isCON], axis=0) for g in range(K)}
Gfc_mean_PD  = {g: np.nanmean(Gfc_sub[g][isPD],  axis=0) for g in range(K)}
print(f'Gfc_sub shape: {Gfc_sub[0].shape}')


In [ ]:
# Save gradient-weighted cortical expression 
import os
OUT_GRAD = results_path + 'gradients/'
os.makedirs(OUT_GRAD, exist_ok=True)

# Save all gradients as npz
np.savez(os.path.join(OUT_GRAD, 'Gfc_sub_all.npz'),
         **{f'G{g+1}_Ctx': Gfc_sub[g] for g in range(K)})
print(f'Saved Gfc_sub_all.npz  shape per gradient: {Gfc_sub[0].shape}')

# Save G1 as CSV with subject IDs
df_g1 = pd.DataFrame(Gfc_sub[0], index=sub_id)
df_g1.index.name = 'SubjectID'
df_g1.to_csv(os.path.join(OUT_GRAD, 'Gfc_sub_G1_parcels.csv'))
print(f'Saved Gfc_sub_G1_parcels.csv  ({df_g1.shape[0]} x {df_g1.shape[1]})')

# Save HC and PD mean templates
pd.DataFrame({'G1_HC_mean': Gfc_mean_CON[0],
              'G1_PD_mean': Gfc_mean_PD[0]}).to_csv(
    os.path.join(OUT_GRAD, 'Gfc_mean_HC_PD_G1.csv'), index=False)
print('Saved Gfc_mean_HC_PD_G1.csv')

# Save group labels
pd.DataFrame({'SubjectID': sub_id,
              'Group': ['Control' if c else 'PD' for c in isCON]}).to_csv(
    os.path.join(OUT_GRAD, 'Gfc_subject_labels.csv'), index=False)
print('Saved Gfc_subject_labels.csv')

In [ ]:
import pandas as pd
import numpy as np

# Save HC normative G1 template (400 parcels)
pd.DataFrame({'CorticalExpression': Gfc_mean_CON[0]}).to_excel(
    results_path + 'controlTemp_G1.xlsx', index=False)
print(f"Saved HC G1 template - controlTemp_G1.xlsx")

# Save PD G1 template
pd.DataFrame({'CorticalExpression': Gfc_mean_PD[0]}).to_excel(
    results_path + 'PD_G1.xlsx', index=False)
print(f"Saved PD G1 template - PD_G1.xlsx")


In [ ]:
# Plot HC mean and PD mean cortical expression on surface
G_cortex_PD  = {}
G_cortex_CON = {}
G_cortex_DIF = {}

# Redefine surface labels and mask (in case overwritten)
surf_labels_lh = nib.freesurfer.read_annot(
    '/path/to/fsa5/lh.Schaefer2018_400Parcels_7Networks_order.annot')[0]
surf_labels_rh = nib.freesurfer.read_annot(
    '/path/to/fsa5/rh.Schaefer2018_400Parcels_7Networks_order.annot')[0]
surf_labels_rh[surf_labels_rh != 0] += 200
surf_labels = np.concatenate([surf_labels_lh, surf_labels_rh])
mask = surf_labels != 0  # boolean array, medial wall = False

for g in range(K):
    vmin_g, vmax_g = get_sym_limits(Gfc_mean_PD[g], Gfc_mean_CON[g])

    G_cortex_CON[g] = map_to_labels(Gfc_mean_CON[g].reshape(nParc), surf_labels, mask=mask, fill=np.nan)
    G_cortex_PD[g]  = map_to_labels(Gfc_mean_PD[g].reshape(nParc),  surf_labels, mask=mask, fill=np.nan)
    G_cortex_DIF[g] = map_to_labels(
        (Gfc_mean_PD[g]-Gfc_mean_CON[g]).reshape(nParc), surf_labels, mask=mask, fill=np.nan)




In [ ]:
# HC mean cortical expression on inflated surface (surfplot)
from sklearn import preprocessing
surfaces = fetch_fsaverage(density='10k')
lh_inf, rh_inf = surfaces['inflated']

BF_lh_data = images.load_gifti(BF_SURF_LH).agg_data()
BF_rh_data = images.load_gifti(BF_SURF_RH).agg_data()
BF_data    = np.concatenate((BF_lh_data, BF_rh_data))
scaler_bf  = preprocessing.MinMaxScaler()

BF_bin     = scaler_bf.fit_transform(BF_data.reshape(-1,1)).flatten()
BF_bin     = np.where(BF_bin == 0.5, 1, BF_bin)

for g in range(K):  
    for label, arr in [('HC', G_cortex_CON[g])]:
        vmin_g, vmax_g = get_sym_limits(G_cortex_CON[g][np.isfinite(G_cortex_CON[g])],
                                        G_cortex_PD[g][np.isfinite(G_cortex_PD[g])])

        p = Plot(lh_inf, rh_inf, size=(800, 600), zoom=1.5, brightness=0.8)
        p.add_layer({'left': arr[:10242], 'right': arr[10242:]},
                    cmap=colors_surf[g], cbar=True,
                    color_range=(vmin_g, vmax_g))
        p.add_layer({'left': BF_bin[:10242], 'right': BF_bin[10242:]},
                    cmap='binary_r', cbar=False)
        fig_p = p.build()
        fig_p.savefig(
            fig_path + f'{label}_G{g+1}_inflated_surfplot.png',
            dpi=300, transparent=True)
        plt.show()
        print(f'Saved: {label} G{g+1} inflated')

## 7. PD vs HC Gradient Comparisons
### 7a. Internal BF gradient structure (range, Ch4/Ch1-3 separation, reference gradient distance)
### 7b. Cortical expression similarity (G1_spearman)
### 7c. Alt similarity metrics (Pearson, cosine, Euclidean)

In [ ]:
seed_vol  = nib.load(BF_SEED).get_fdata()
atlas_vol = nib.load(BF_ATLAS).get_fdata()
bf_mask   = seed_vol != 0
atlas_vox = atlas_vol[bf_mask].flatten()
ch123_m = atlas_vox == 1
ch4_m   = atlas_vox == 2
hc_template_g1 = np.nanmean(G_ind[isCON,:,0], axis=0)
G_ref  = gm.gradients_.copy()  # whole-sample template
G_ref_G1 = G_ref[:,0]
metrics = []
for s in range(nSub):
    g1 = G_ind[s,:,0]
    metrics.append({'SubjectID':sub_id[s],'Group':group[s],
        'G1_range':      g1.max()-g1.min(),
        'Ch4_Ch123_sep': g1[ch4_m].mean()-g1[ch123_m].mean(),
        'ref_grad_distance':  np.sqrt(np.nansum((g1-G_ref_G1)**2))})
df_bf = pd.DataFrame(metrics)
print(df_bf.groupby('Group')[['G1_range','Ch4_Ch123_sep','ref_grad_distance']].agg(['mean','std']).round(4))


In [ ]:
import statsmodels.api as smapi
from statsmodels.stats.multitest import multipletests
# HC-template G1_pearson: subject cortical expression correlated with HC normative cortical expression
cov = pd.read_excel(COV_FILE)
# Create sub_int fresh
df_bf['sub_int'] = df_bf['SubjectID'].str.replace('sub-','').astype(int)

cov_merge = cov[['PATNO','Age','Sex']].copy()
cov_merge['PATNO'] = cov_merge['PATNO'].astype(int)
df_bf = df_bf.merge(cov_merge, left_on='sub_int', right_on='PATNO', how='left')
print(f"Merge: {len(df_bf)} rows, Age nulls: {df_bf['Age'].isna().sum()}")

grp_b = (df_bf['Group']=='PD').astype(float).values
age_v = df_bf['Age'].values.astype(float)
sex_v = df_bf['Sex'].values.astype(float)

print('Internal BF structure — PD vs HC:')
bf_results = []
for metric in ['G1_range','Ch4_Ch123_sep','ref_grad_distance']:
    y = df_bf[metric].values.astype(float)
    X = smapi.add_constant(np.column_stack([grp_b, age_v, sex_v]))
    model = smapi.OLS(y, X).fit()
    b, t, p = model.params[1], model.tvalues[1], model.pvalues[1]
    pd_v = y[df_bf['Group']=='PD']
    hc_v = y[df_bf['Group']=='Control']
    d = (pd_v.mean() - hc_v.mean()) / np.sqrt((pd_v.var(ddof=1) + hc_v.var(ddof=1)) / 2)
    print(f'  {metric:20s}: beta={b:.4f}, t={t:.3f}, p={p:.4f}, d={d:.3f}')
    bf_results.append({'metric': metric, 'beta': b, 't': t, 'p': p, 'cohens_d': d,
                       'HC_mean': hc_v.mean(), 'HC_sd': hc_v.std(),
                       'PD_mean': pd_v.mean(), 'PD_sd': pd_v.std()})

df_bf_res = pd.DataFrame(bf_results)
_, df_bf_res['p_fdr'], _, _ = multipletests(df_bf_res['p'], method='fdr_bh')
print('\nFDR results:')
print(df_bf_res[['metric','beta','t','p','p_fdr','cohens_d']].to_string(index=False))
df_bf_res.to_csv(results_path + 'BF_internal_structure_PD_HC.csv', index=False)


In [ ]:
# Quick plot
fig, axes = plt.subplots(1, 3, figsize=(12,4))
palette_grp = {'Control':'#2196A6','PD':'#CD3E4E'}
for ax, metric, label in zip(axes,
    ['G1_range','Ch4_Ch123_sep','ref_grad_distance'],
    ['G1 range','Ch4 - Ch1-3 mean G1','Reference-Gradient distance']):
    sns.violinplot(data=df_bf, x='Group', y=metric, order=['Control','PD'],
                   palette=palette_grp, inner='box', ax=ax, linewidth=1)
    p = df_bf_res.loc[df_bf_res['metric']==metric,'p'].values[0]
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
    ax.set_title(f'{label}\np={p:.3f} ({sig})', fontsize=9)
    ax.set_xlabel('')
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(fig_path + 'BF_internal_structure_PD_HC.pdf', dpi=300, bbox_inches='tight')
plt.savefig(fig_path + 'BF_internal_structure_PD_HC.png', dpi=300, bbox_inches='tight')
plt.show()


### 7b. Cortical expression similarity (Primary: G1_spearman)

In [ ]:
from scipy.stats import spearmanr

G1_spearman = np.array([
    spearmanr(Gfc_sub[0][s], Gfc_mean_CON[0], nan_policy='omit')[0]
    for s in range(nSub)
])

cov = pd.read_excel(COV_FILE)
cov['PATNO'] = cov['PATNO'].astype(int)
sub_id_int = np.array([int(s.replace('sub-','')) for s in sub_id])
df_metrics = pd.DataFrame({'SubjectID':sub_id,'Group':group,'G1_spearman':G1_spearman})
df_metrics = df_metrics.merge(cov[['PATNO','Age','Sex']], left_on=sub_id_int, right_on='PATNO', how='left')
df_metrics.to_csv(results_path + 'gradient_subject_metrics_spearman.csv', index=False)
print(f'HC-template sensitivity | HC G1_spearman: {G1_spearman[isCON].mean():.3f} ± {G1_spearman[isCON].std():.3f}')
print(f'HC-template sensitivity | PD G1_spearman: {G1_spearman[isPD].mean():.3f} ± {G1_spearman[isPD].std():.3f}')



In [ ]:
def run_model(y, grp, age, sex, label=''):
    y_use = np.arctanh(np.clip(y, -0.9999, 0.9999))
    X = smapi.add_constant(np.column_stack([grp, age, sex]))
    model = smapi.OLS(y_use, X).fit()
    b, t, p = model.params[1], model.tvalues[1], model.pvalues[1]
    d = (y_use[isPD].mean() - y_use[isCON].mean()) / \
        np.sqrt((y_use[isPD].var(ddof=1) + y_use[isCON].var(ddof=1)) / 2)
    print(f'{label:30s}: beta={b:.4f}, t={t:.3f}, p={p:.4f}, d={d:.3f}')
    return {'label': label, 'beta': b, 't': t, 'p': p, 'cohens_d': d}

cov_df = df_metrics.dropna(subset=['Age','Sex'])
grp_b  = (cov_df['Group']=='PD').astype(float).values
age_c  = cov_df['Age'].values.astype(float)
sex_c  = cov_df['Sex'].values.astype(float)
isPD   = cov_df['Group'].values == 'PD'
isCON  = cov_df['Group'].values == 'Control'

print('Cortical alignment — PD vs HC:')
results_align = []
for g, label in enumerate(['G1 alignment']):
    col = ['G1_spearman'][g]
    results_align.append(run_model(cov_df[col].values, grp_b, age_c, sex_c, label))
pd.DataFrame(results_align).to_csv(results_path + 'PD_HC_alignment_results.csv', index=False)

In [ ]:
from scipy.stats import levene
stat, p = levene(cov_df.loc[cov_df['Group']=='PD', 'G1_spearman'],
                 cov_df.loc[cov_df['Group']=='Control', 'G1_spearman'])
print(f"Levene test: F={stat:.3f}, p={p:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
palette_g = {'Control': '#7D3C98', 'PD': '#2196A6'}

for grp_name, col in palette_g.items():
    m   = group == grp_name
    x0  = 0 if grp_name == 'Control' else 1
    ax.scatter(np.random.normal(x0, 0.07, m.sum()),
               G1_spearman[m], alpha=0.5, s=20, color=col,
               label=grp_name, zorder=3)
    ax.hlines(G1_spearman[m].mean(),
              x0 - 0.4, x0 + 0.4,
              color=col, lw=2.5, zorder=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(['HC', 'PD'], fontsize=12)
ax.set_ylim(-0.4, 0.75)
ax.text(0.98, 0.98, f'p = 0.015\nd = −0.31',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=9, color='#333333')
ax.set_ylabel('G1 alignment (Spearmans rho with HC mean Gfc)', fontsize=10)
ax.set_title('G1 alignment: PD vs HC', fontsize=11)
ax.legend(fontsize=9, frameon=False, loc='lower left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(fig_path + 'G1_alignment_spearman_PD_HC.pdf', dpi=300, bbox_inches='tight')
plt.savefig(fig_path + 'G1_alignment_spearman_PD_HC.png', dpi=300, bbox_inches='tight')
plt.show()

### 7c. Alt similarity metrics

In [ ]:
from scipy.stats import rankdata
from scipy.spatial.distance import cosine, euclidean

def row_corr(A, v, eps=1e-12):
    A = np.asarray(A, float)
    v = np.asarray(v, float).reshape(-1)

    A0 = A - np.nanmean(A, axis=1, keepdims=True)
    v0 = v - np.nanmean(v)

    num = np.nansum(A0 * v0, axis=1)
    den = np.sqrt(np.nansum(A0**2, axis=1) * np.nansum(v0**2))

    out = num / (den + eps)
    out[~np.isfinite(out)] = np.nan
    return out


def row_spearman(A, v, eps=1e-12):
    A = np.asarray(A, float)
    v = np.asarray(v, float).reshape(-1)

    out = np.full(A.shape[0], np.nan)

    for i in range(A.shape[0]):
        vec = A[i, :]

        ok = np.isfinite(vec) & np.isfinite(v)

        if ok.sum() < 5:
            continue

        vec_rank = rankdata(vec[ok])
        v_rank = rankdata(v[ok])

        out[i] = row_corr(vec_rank.reshape(1, -1), v_rank)[0]

    return out


sim_pearson  = np.zeros((nSub, K))
sim_spearman = np.zeros((nSub, K))
sim_cosine   = np.zeros((nSub, K))
dist_euclid  = np.zeros((nSub, K))

for g in range(K):

    template = np.asarray(Gfc_mean_CON[g]).reshape(-1)

    sim_pearson[:, g] = row_corr(Gfc_sub[g], template)
    sim_spearman[:, g] = row_spearman(Gfc_sub[g], template)

    for i in range(nSub):
        vec = np.asarray(Gfc_sub[g][i]).reshape(-1)

        ok = np.isfinite(vec) & np.isfinite(template)

        if ok.sum() < 5:
            sim_cosine[i, g] = np.nan
            dist_euclid[i, g] = np.nan
        else:
            sim_cosine[i, g] = 1 - cosine(vec[ok], template[ok])
            dist_euclid[i, g] = euclidean(vec[ok], template[ok])


print('\nSimilarity to HC Gfc:')

for sim_arr, sim_label in [
    (sim_pearson,  'Pearson'),
    (sim_spearman, 'Spearman'),
    (sim_cosine,   'Cosine'),
    (dist_euclid,  'Euclidean dist')
]:

    print(f'\n  {sim_label}:')

    for g in range(K):

        con_vals = sim_arr[isCON, g]
        pd_vals  = sim_arr[isPD, g]

        print(
            f'    G{g+1}: '
            f'HC={np.nanmean(con_vals):+.3f} ± {np.nanstd(con_vals):.3f}, '
            f'PD={np.nanmean(pd_vals):+.3f} ± {np.nanstd(pd_vals):.3f}'
        )

        run_model(
            sim_arr[:, g],
            isPD.astype(float),
            df_metrics['Age'].values.astype(float),
            df_metrics['Sex'].values.astype(float),
            label=f'    G{g+1}'
        )

## 8. Check robustness of HC template

In [ ]:
# Quick check on robustness of HC template

# Config
G = 0  # G1
N_BOOT = 10000
RNG_SEED = 1234

rng = np.random.default_rng(RNG_SEED)

G1_maps = np.asarray(Gfc_sub[G], dtype=float)   # nSub x 400
hc_idx = np.where(isCON)[0]
pd_idx = np.where(isPD)[0]

orig_HC_template = np.nanmean(G1_maps[hc_idx, :], axis=0)

# Redefine helpers

def row_spearman(A, v):
    out = np.full(A.shape[0], np.nan)
    for i in range(A.shape[0]):
        ok = np.isfinite(A[i, :]) & np.isfinite(v)
        if ok.sum() > 10:
            out[i] = spearmanr(A[i, ok], v[ok])[0]
    return out

def vec_spearman(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    return spearmanr(a[ok], b[ok])[0]

def vec_pearson(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    return pearsonr(a[ok], b[ok])[0]

# Leave one out template stability

loo_template_corr = []
loo_subject_alignment = []

for h in hc_idx:

    loo_hc = hc_idx[hc_idx != h]
    loo_template = np.nanmean(G1_maps[loo_hc, :], axis=0)

    loo_template_corr.append(vec_spearman(loo_template, orig_HC_template))

    # alignment of all subjects to this LOO template
    loo_align = row_spearman(G1_maps, loo_template)
    loo_subject_alignment.append(loo_align)

loo_template_corr = np.asarray(loo_template_corr)
loo_subject_alignment = np.vstack(loo_subject_alignment)  # nHC x nSub

mean_loo_alignment = np.nanmean(loo_subject_alignment, axis=0)
sd_loo_alignment   = np.nanstd(loo_subject_alignment, axis=0)

orig_alignment = row_spearman(G1_maps, orig_HC_template)

stability_df = pd.DataFrame({
    "SubjectID": sub_id,
    "Group": group,
    "G1_spearman_original_HC_template": orig_alignment,
    "G1_spearman_LOO_mean": mean_loo_alignment,
    "G1_spearman_LOO_sd": sd_loo_alignment,
})

stability_df.to_csv(
    results_path + "G1_LOO_HC_template_subject_alignment.csv",
    index=False
)

print("\nLOO HC template stability")
print(f"Template similarity to original HC template:")
print(f"  mean rho = {np.nanmean(loo_template_corr):.4f}")
print(f"  SD rho   = {np.nanstd(loo_template_corr):.4f}")
print(f"  min rho  = {np.nanmin(loo_template_corr):.4f}")
print(f"  max rho  = {np.nanmax(loo_template_corr):.4f}")

print("\nSubject alignment stability across LOO templates")
print(f"  mean SD across all subjects = {np.nanmean(sd_loo_alignment):.4f}")
print(f"  mean SD in PD subjects      = {np.nanmean(sd_loo_alignment[pd_idx]):.4f}")
print(f"  mean SD in HC subjects      = {np.nanmean(sd_loo_alignment[hc_idx]):.4f}")

# Bootstrapping

boot_template_corr = np.full(N_BOOT, np.nan)
boot_PD_mean_align = np.full(N_BOOT, np.nan)
boot_HC_mean_align = np.full(N_BOOT, np.nan)

for b in range(N_BOOT):

    boot_hc = rng.choice(hc_idx, size=len(hc_idx), replace=True)
    boot_template = np.nanmean(G1_maps[boot_hc, :], axis=0)

    boot_template_corr[b] = vec_spearman(boot_template, orig_HC_template)

    boot_align = row_spearman(G1_maps, boot_template)

    boot_PD_mean_align[b] = np.nanmean(boot_align[pd_idx])
    boot_HC_mean_align[b] = np.nanmean(boot_align[hc_idx])

boot_df = pd.DataFrame({
    "bootstrap_iter": np.arange(1, N_BOOT + 1),
    "template_rho_with_original": boot_template_corr,
    "PD_mean_alignment": boot_PD_mean_align,
    "HC_mean_alignment": boot_HC_mean_align,
    "HC_minus_PD_alignment": boot_HC_mean_align - boot_PD_mean_align,
})

boot_df.to_csv(
    results_path + "G1_bootstrap_HC_template_stability.csv",
    index=False
)

print("\nBootstrap HC template stability")
print(f"Template rho with original:")
print(f"  mean = {np.nanmean(boot_template_corr):.4f}")
print(f"  95% CI = [{np.nanpercentile(boot_template_corr, 2.5):.4f}, {np.nanpercentile(boot_template_corr, 97.5):.4f}]")

print("\nBootstrap group separation")
print(f"HC - PD alignment difference:")
print(f"  mean = {np.nanmean(boot_HC_mean_align - boot_PD_mean_align):.4f}")
print(f"  95% CI = [{np.nanpercentile(boot_HC_mean_align - boot_PD_mean_align, 2.5):.4f}, {np.nanpercentile(boot_HC_mean_align - boot_PD_mean_align, 97.5):.4f}]")


# Quick figures

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

axes[0].hist(loo_template_corr, bins=20, color="#7B4F9E", alpha=0.8)
axes[0].set_title("LOO HC template")
axes[0].set_xlabel("ρ with original HC template")
axes[0].set_ylabel("Count")

axes[1].hist(boot_template_corr, bins=40, color="#2D9B8A", alpha=0.8)
axes[1].set_title("Bootstrap HC template")
axes[1].set_xlabel("ρ with original HC template")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(results_path + "G1_template_stability_LOO_bootstrap.png", dpi=300, bbox_inches="tight")
plt.savefig(results_path + "G1_template_stability_LOO_bootstrap.pdf", bbox_inches="tight")
plt.show()